In [1]:
# ============================================================
# Imports
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import models

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# Project Directories
# ============================================================

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"

PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR = DATA_DIR / "raw"

MODEL_DIR = PROJECT_DIR / "models"

OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR.mkdir(exist_ok=True)

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
# ============================================================
# Configuration
# ============================================================

IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_EPOCHS = 20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

RANDOM_SEED = 42

In [4]:
# ============================================================
# Random Seed
# ============================================================

random.seed(RANDOM_SEED)

np.random.seed(RANDOM_SEED)

torch.manual_seed(RANDOM_SEED)

torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

print("Random seed fixed.")

Random seed fixed.


In [5]:
# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device :", DEVICE)

Device : cpu


In [6]:
# ============================================================
# Load Metadata
# ============================================================

train_df = pd.read_csv(
    PROCESSED_DIR / "train_metadata.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "validation_metadata.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test_metadata.csv"
)

print(len(train_df))
print(len(val_df))
print(len(test_df))

78566
16106
17448


In [7]:
# ============================================================
# Load Disease Classes
# ============================================================

with open(PROCESSED_DIR / "disease_classes.json", "r") as f:
    label_columns = json.load(f)

print("=" * 60)
print("Disease Classes")
print("=" * 60)

print(f"Total Classes : {len(label_columns)}\n")

for i, disease in enumerate(label_columns, start=1):
    print(f"{i:02d}. {disease}")

Disease Classes
Total Classes : 15

01. Atelectasis
02. Cardiomegaly
03. Consolidation
04. Edema
05. Effusion
06. Emphysema
07. Fibrosis
08. Hernia
09. Infiltration
10. Mass
11. No Finding
12. Nodule
13. Pleural_Thickening
14. Pneumonia
15. Pneumothorax


In [8]:
# ============================================================
# Image Lookup Dictionary
# ============================================================

IMAGE_DIR = RAW_DIR / "images"

image_lookup = {}

image_folders = sorted(IMAGE_DIR.glob("images_*"))

for folder in image_folders:

    actual_folder = folder / "images"

    for img_path in actual_folder.glob("*.png"):

        image_lookup[img_path.name] = img_path

print("=" * 60)
print("Image Lookup Created")
print("=" * 60)

print(f"Total Images Found : {len(image_lookup):,}")

Image Lookup Created
Total Images Found : 112,120


In [9]:
# ============================================================
# Image Transformations
# ============================================================

from torchvision import transforms

train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomRotation(7),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )

])

val_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )

])

test_transform = val_transform

print("Transforms Ready")

Transforms Ready


In [10]:
# ============================================================
# Custom Dataset
# ============================================================

from torch.utils.data import Dataset

class ChestXrayDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_lookup,
        label_columns,
        transform=None
    ):

        self.df = dataframe.reset_index(drop=True)

        self.image_lookup = image_lookup

        self.transform = transform

        self.image_names = self.df["Image Index"].tolist()

        self.labels = self.df[label_columns].to_numpy(dtype=np.float32)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.image_lookup[self.image_names[idx]]

        image = Image.open(image_path).convert("L")

        if self.transform:

            image = self.transform(image)

        label = torch.from_numpy(self.labels[idx])

        return image, label

In [11]:
# ============================================================
# Dataset Objects
# ============================================================

train_dataset = ChestXrayDataset(
    train_df,
    image_lookup,
    label_columns,
    train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_lookup,
    label_columns,
    val_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_lookup,
    label_columns,
    test_transform
)

print("=" * 60)
print("Datasets Created")
print("=" * 60)

print("Train :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test :", len(test_dataset))

Datasets Created
Train : 78566
Validation : 16106
Test : 17448


In [12]:
# ============================================================
# Positive Class Weights
# ============================================================

positive_counts = train_df[label_columns].sum()

negative_counts = len(train_df) - positive_counts

pos_weights = negative_counts / positive_counts

pos_weights = torch.tensor(
    pos_weights.values,
    dtype=torch.float32
).to(DEVICE)

print("Positive Class Weights Ready")

Positive Class Weights Ready


In [13]:
# ============================================================
# Weighted Random Sampler
# ============================================================

from torch.utils.data import WeightedRandomSampler

sample_weights = train_df[label_columns].sum(axis=1)

sample_weights = sample_weights.replace(0, 1)

weighted_sampler = WeightedRandomSampler(

    weights=torch.DoubleTensor(sample_weights),

    num_samples=len(sample_weights),

    replacement=True
)

print("Weighted Sampler Ready")

Weighted Sampler Ready


In [14]:
# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    sampler=weighted_sampler,

    num_workers=0,

    pin_memory=False

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=False

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=False

)

print("DataLoaders Created")

DataLoaders Created


In [15]:
# ============================================================
# Verify Data Pipeline
# ============================================================

images, labels = next(iter(train_loader))

print("=" * 60)

print("Batch Verification")

print("=" * 60)

print("Images Shape :", images.shape)

print("Labels Shape :", labels.shape)

Batch Verification
Images Shape : torch.Size([32, 1, 224, 224])
Labels Shape : torch.Size([32, 15])
